# 10. SOLECO electricity load profiles

Survey the 2018–2026 delivery by workbook and worksheet, combine meter readings at their recorded timestamps, then inspect coverage and delivered load. Run the cells in order. Section 1 writes the layout inventory before Section 2 imports measurements. Excel readers require `openpyxl` for XLSX/XLSM and `xlrd` for legacy XLS. HTML exports with an XLS extension use `lxml`.

The record date comes from the worksheet. For example, a January delivery can contain December readings. The two confirmed meter-ID pairs are combined into one series per pair. Original IDs remain available in the exports. The cooperative’s coincidental total is retained as a separate series.

In [ ]:
from pathlib import Path
import csv
import re

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
SOLECO_DIR = PROJECT_DIR / "datasets" / "Soleco Load Profile"
OUTPUT_DIR = PROJECT_DIR / "datasets" / "soleco_processed"
DAY_FIRST = False  # Change to True if the source uses DD/MM/YYYY.
HEADER_SCAN_ROWS = 60
GAP_START = pd.Timestamp("2022-01-01")
GAP_END = pd.Timestamp("2023-05-26")
SERIES_ALIASES = {
    "MF3MMAASOLE01": "MF3MORCSOLE01",
    "MF3MMAASOLE02": "MF3MORCSOLE02",
}


# Optional edits after inspecting the inventory. Keys use the raw workbook wording.
HEADER_ALIASES = {}   # Example: {"Interval End": "DATETIME"}
SHEET_IDS = {}        # Example: {("2018/Jan 2018/Jan NGCP.xls", "Sheet1"): "verified ID"}
METER_FIELDS = ("KW_DEL", "KWH_DEL", "KVARH_DEL", "KW_REC", "KWH_REC", "KVARH_REC")
MEASURE_FIELDS = [*METER_FIELDS, "reported_load"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(items, desc):
        for number, item in enumerate(items, 1):
            print(f"{desc}: {number}/{len(items)}", end="\r", flush=True)
            yield item

## 1. Survey the archive

Inspect every Excel tab and the first 60 rows of each sheet or CSV. The inventory records the original headers, their interpreted fields, unrecognised columns and a preview of unfamiliar layouts. Increase `HEADER_SCAN_ROWS` or add an alias above if the inventory shows a different convention.

In [ ]:
if not SOLECO_DIR.is_dir():
    raise FileNotFoundError(f"Set SOLECO_DIR to your SOLECO folder. Current path: {SOLECO_DIR}")

supported = {".xlsx", ".xlsm", ".xls", ".csv"}
files = sorted(
    path for path in SOLECO_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in supported
    and not any(part.startswith((".", "~$")) for part in path.relative_to(SOLECO_DIR).parts)
)
if not files:
    raise FileNotFoundError(f"No supported load files found under {SOLECO_DIR}")


from contextlib import contextmanager
from io import BytesIO
from zipfile import is_zipfile

def token(value):
    return re.sub(r"[^A-Z0-9]+", "_", str(value).strip().upper()).strip("_")

def meter_id(value):
    if pd.isna(value):
        return None
    return re.sub(r"^\d+\s*-\s*", "", str(value).strip().upper()) or None

def series_header(value):
    value = meter_id(value) or ""
    if token(value) in {"TOTAL", "TOTAL_LOAD", "TOTAL_COINCIDENTAL_LOAD", "TOTAL_COINCIDENT_LOAD"}:
        return "TOTAL"
    if re.fullmatch(r"MF[0-9A-Z]+", value) and any(c.isdigit() for c in value):
        return value
    for station in ("Sogod", "Saint Bernard", "Maasin", "Himay-angan"):
        if token(value) in {token(station), token(station) + "_SUBSTATION"}:
            return station
    return None

aliases = {
    "BDATE": "DATE", "B_DATE": "DATE", "BILLING_DATE": "DATE",
    "DATE_TIME": "DATETIME", "TIMESTAMP": "DATETIME", "DATE_AND_TIME": "DATETIME",
    "SEIN": "SEIL", "METER_ID": "SEIL", "SITE_AND_EQUIPMENT_IDENTIFICATION_AND_LABELLING": "SEIL",
}
aliases.update({token(k): v for k, v in HEADER_ALIASES.items()})
def field(value):
    name = token(value)
    name = aliases.get(name, name)
    if name in {"DATE", "TIME", "DATETIME", "SEIL", *METER_FIELDS}:
        return name
    return series_header(name) or series_header(value)

def workbook_format(path):
    if path.suffix.lower() == ".csv":
        return "csv"
    with path.open("rb") as handle:
        signature = handle.read(4096)
    # Legacy Excel workbooks and newer workbooks exported with an XLS extension.
    if signature.startswith((b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1",
                             b"\x09\x00\x04\x00", b"\x09\x02\x06\x00",
                             b"\x09\x04\x06\x00", b"\x09\x08\x08\x00")):
        return "xlrd"
    markup = signature.replace(b"\x00", b"").lower()
    if b"<html" in markup or b"<table" in markup or b"<!doctype html" in markup:
        return "html"
    if path.suffix.lower() == ".xls" and is_zipfile(path):
        return "openpyxl"
    return "auto"


@contextmanager
def open_source(path, source_format):
    if source_format == "html":
        tables = pd.read_html(BytesIO(path.read_bytes()), flavor="lxml", header=None)
        rows_by_sheet = {}
        for number, table in enumerate(tables, 1):
            # read_html promotes HTML header rows to column labels.
            headings = table.columns.to_frame(index=False).T.to_numpy()
            raw = table.copy()
            raw.columns = range(raw.shape[1])
            if not isinstance(table.columns, pd.RangeIndex):
                raw = pd.concat([pd.DataFrame(headings), raw], ignore_index=True)
            rows_by_sheet[f"HTML table {number}"] = raw
        yield list(rows_by_sheet), lambda sheet, nrows=None: rows_by_sheet[sheet].iloc[:nrows]
        return
    if source_format != "csv":
        engine = None if source_format == "auto" else source_format
        with pd.ExcelFile(path, engine=engine) as book:
            yield book.sheet_names, lambda sheet, nrows=None: book.parse(
                sheet, header=None, dtype=object, nrows=nrows
            )
        return
    try:
        preview = path.open(encoding="utf-8-sig")
        with preview:
            sample = preview.read(8192)
        encoding = "utf-8-sig"
    except UnicodeDecodeError:
        encoding = "cp1252"
        with path.open(encoding=encoding) as handle:
            sample = handle.read(8192)
    try:
        delim = csv.Sniffer().sniff(sample, delimiters=",;\t").delimiter
    except csv.Error:
        delim = max(",;\t", key=lambda char: sample.count(char))
    def read_csv(sheet, nrows=None):
        from itertools import islice
        with path.open(encoding=encoding, newline="") as handle:
            return pd.DataFrame(list(islice(csv.reader(handle, delimiter=delim), nrows)))
    yield ["CSV"], read_csv

def inspect_sheet(raw, source, sheet):
    sample_rows = [" | ".join(str(v) for v in row if pd.notna(v) and str(v).strip())
                   for row in raw.head(4).itertuples(index=False, name=None)]
    info = dict(source_file=source, source_sheet=sheet, status="unrecognised layout",
                preview=" / ".join(sample_rows), raw_headers="", fields="", extra_columns="")
    fallback = SHEET_IDS.get((source, sheet), series_header(sheet))
    for pos, row in enumerate(raw.itertuples(index=False, name=None)):
        names = [field(v) if pd.notna(v) else None for v in row]
        if not {"DATE", "DATETIME"}.intersection(names):
            continue
        known = [v for v in names if v]
        meter = any(v in METER_FIELDS for v in known)
        wide = [v for v in known if v not in {"DATE", "DATETIME", "TIME", "SEIL", *METER_FIELDS}]
        info.update(header_row=pos + 1, raw_headers=" | ".join(str(v) for v in row if pd.notna(v)),
                    fields=" | ".join(known), extra_columns=" | ".join(
                        str(v) for v, name in zip(row, names) if pd.notna(v) and not name))
        if not meter and not wide:
            continue
        if len(set(known)) != len(known) or (meter and wide):
            info["status"] = "ambiguous fields"
            return info, None
        if meter and "SEIL" not in known and not fallback:
            info["status"] = "meter ID needed"
            return info, None
        layout = "meter" if meter else "profile"
        info.update(status="ready", layout=layout)
        return info, (pos, names, layout, fallback)
    return info, None

survey, plans, formats = [], {}, {}
for path in tqdm(files, desc="Survey files"):
    source = path.relative_to(SOLECO_DIR).as_posix()
    plans[path] = {}
    try:
        formats[path] = workbook_format(path)
        with open_source(path, formats[path]) as (sheets, read):
            for sheet in sheets:
                try:
                    info, plan = inspect_sheet(read(sheet, HEADER_SCAN_ROWS), source, sheet)
                    survey.append(info | {"source_format": formats[path]})
                    if plan:
                        plans[path][sheet] = plan
                except Exception as exc:
                    survey.append(dict(source_file=source, source_sheet=sheet, source_format=formats[path], status=f"Preview failed: {exc}"))
    except Exception as exc:
        survey.append(dict(source_file=source, source_sheet="", source_format=formats.get(path), status=f"File failed: {exc}"))

inventory = pd.DataFrame(survey).reindex(columns=[
    "source_file", "source_sheet", "source_format", "status", "layout", "header_row", "raw_headers",
    "fields", "extra_columns", "preview",
])
inventory.to_csv(OUTPUT_DIR / "soleco_sheet_inventory.csv", index=False)
patterns = inventory.groupby(["status", "layout", "raw_headers"], dropna=False).size().rename("sheets")
display(patterns.to_frame())
display(inventory.loc[inventory.status.ne("ready") | inventory.extra_columns.fillna("").ne(""),
                      ["source_file", "source_sheet", "status", "extra_columns", "preview"]])
if inventory.status.str.contains(r"xlrd|openpyxl|lxml", case=False, regex=True).any():
    print("Check the reader errors above. In this notebook kernel, install missing readers "
          "with %pip install xlrd openpyxl lxml, then rerun from Section 1.")


## 2. Import readings and audit meter identities

Use each sheet’s detected header row and column order. Read meter IDs from the records, using an identifiable tab name for missing IDs. The ID audit shows which series actually occur on each sheet. Wide-format load values retain their original series labels and an unspecified unit.

In [ ]:
def parse_excel_datetime(values):
    text = values.astype("string").str.strip()
    numeric = pd.to_numeric(text, errors="coerce")
    serial = numeric.between(20000, 100000).fillna(False)
    result = pd.to_datetime(text.mask(serial), format="mixed", errors="coerce", dayfirst=DAY_FIRST)
    result.loc[serial] = pd.to_datetime(numeric.loc[serial], origin="1899-12-30", unit="D")
    return result


def parse_timestamps(frame):
    if "DATETIME" in frame:
        return parse_excel_datetime(frame["DATETIME"])

    dates = parse_excel_datetime(frame["DATE"])
    if "TIME" not in frame:
        return dates

    text = frame["TIME"].astype("string").str.strip()
    numeric = pd.to_numeric(text, errors="coerce")
    fraction = (numeric.ge(0) & numeric.lt(1)).fillna(False)
    # Excel fractions and textual times such as 01:00, 01:00:00, or 1:00 AM.
    time_only = text.str.replace(r"^\d{4}-\d\d-\d\d[ T]", "", regex=True)
    parsed = pd.to_datetime("2000-01-01 " + time_only.mask(fraction),
                            format="mixed", errors="coerce")
    elapsed = parsed - pd.Timestamp("2000-01-01")
    elapsed.loc[fraction] = pd.to_timedelta(numeric.loc[fraction], unit="D").dt.round("s")
    end_of_day = time_only.str.fullmatch("24:00(?::00)?", na=False)
    elapsed.loc[end_of_day] = pd.Timedelta(days=1)
    # Keep date-only rows from receiving a fabricated timestamp in a Date + Time sheet.
    return dates.dt.normalize() + elapsed


def numeric_values(values):
    return pd.to_numeric(values.astype("string").str.strip().str.replace(",", "", regex=False),
                         errors="coerce")



records, audit = [], []
for path in tqdm(files, desc="Import files"):
    if not plans[path]:
        continue
    source = path.relative_to(SOLECO_DIR).as_posix()
    try:
        with open_source(path, formats[path]) as (_, read):
            for sheet, (pos, names, layout, fallback) in plans[path].items():
                try:
                    raw = read(sheet)
                    columns = [i for i, name in enumerate(names) if name]
                    frame = raw.iloc[pos + 1:, columns].copy()
                    frame.columns = [names[i] for i in columns]
                    frame["source_row"] = frame.index + 1
                    frame["timestamp"] = parse_timestamps(frame)
                    ids = ["timestamp", "source_row"]
                    if layout == "meter":
                        measures = [k for k in METER_FIELDS if k in frame]
                        data = frame[ids + measures].copy()
                        ids_raw = frame["SEIL"].map(meter_id) if "SEIL" in frame else pd.Series(None, index=frame.index, dtype=object)
                        data["series_id"] = "TOTAL" if fallback == "TOTAL" else ids_raw.where(ids_raw.notna(), fallback)
                    else:
                        wide = [k for k in names if k and k not in {"DATE", "DATETIME", "TIME", "SEIL"}]
                        data = frame.melt(id_vars=ids, value_vars=wide, var_name="series_id", value_name="reported_load")
                        measures = ["reported_load"]
                    populated = data[measures].apply(lambda col: col.astype("string").str.strip().ne("").fillna(False))
                    data[measures] = data[measures].apply(numeric_values)
                    bad_numbers = int((populated & data[measures].isna()).sum().sum())
                    invalid_dates = int((data.timestamp.isna() & populated.any(axis=1)).sum())
                    missing_ids = int((data.series_id.isna() & populated.any(axis=1)).sum())
                    data = data.dropna(subset=["timestamp", "series_id"]).dropna(subset=measures, how="all")
                    if len(data):
                        records.append(data.assign(layout=layout, source_file=source, source_sheet=sheet))
                    audit.append((source, sheet, len(data), invalid_dates, missing_ids, bad_numbers,
                                  "parsed" if len(data) else "no usable readings"))
                except Exception as exc:
                    audit.append((source, sheet, 0, 0, 0, 0, f"Import failed: {exc}"))
    except Exception as exc:
        audit.extend((source, sheet, 0, 0, 0, 0, f"File failed: {exc}") for sheet in plans[path])

import_audit = inventory.merge(pd.DataFrame(audit, columns=[
    "source_file", "source_sheet", "usable_records", "invalid_timestamp_rows",
    "missing_ids", "invalid_numeric_cells", "import_status",
]), on=["source_file", "source_sheet"], how="left")
import_audit.to_csv(OUTPUT_DIR / "soleco_import_audit.csv", index=False)
if not records:
    raise ValueError("No usable readings. Review soleco_sheet_inventory.csv and soleco_import_audit.csv.")
combined_raw = pd.concat(records, ignore_index=True).reindex(columns=[
    "timestamp", "series_id", "layout", *MEASURE_FIELDS, "source_file", "source_sheet", "source_row",
])
del records
id_audit = combined_raw.groupby(["source_file", "source_sheet", "series_id"]).agg(
    records=("timestamp", "size"), first=("timestamp", "min"), last=("timestamp", "max")
).reset_index()
id_audit = id_audit.rename(columns={"series_id": "original_series_id"})
id_audit["series_id"] = id_audit.original_series_id.replace(SERIES_ALIASES)
id_audit.to_csv(OUTPUT_DIR / "soleco_meter_id_inventory.csv", index=False)
display(id_audit.groupby("series_id").agg(files=("source_file", "nunique"), first=("first", "min"), last=("last", "max")))

## 3. Combine and export

Apply the confirmed meter-ID pairs before checking overlaps by layout, series and timestamp. Collapse identical measurements. Set aside conflicting keys with their source rows. Keep measured zeros. The gap checks compare parsed and retained delivered-load readings and report the source worksheets for the affected period. Export the combined table and coverage summary under `datasets/soleco_processed/`.

In [ ]:
key = ["layout", "series_id", "timestamp"]
combined_raw["original_series_id"] = combined_raw["series_id"]
combined_raw["series_id"] = combined_raw["series_id"].replace(SERIES_ALIASES)
combined_raw["series_name"] = combined_raw["series_id"].replace({"TOTAL": "Total coincidental load"})
distinct = combined_raw.drop_duplicates(subset=key + MEASURE_FIELDS)
conflict_keys = distinct.loc[distinct.duplicated(subset=key, keep=False), key].drop_duplicates()
conflicts = combined_raw.merge(conflict_keys, on=key).sort_values(key)
combined = (
    distinct.drop_duplicates(subset=key, keep=False)
    .sort_values(["timestamp", "layout", "series_id"])
    .loc[:, ["timestamp", "series_id", "series_name", "original_series_id", "layout", *MEASURE_FIELDS,
             "source_file", "source_sheet", "source_row"]]
    .reset_index(drop=True)
)

coverage = (
    combined.groupby(["layout", "series_id", "series_name"])
    .agg(records=("timestamp", "size"), first=("timestamp", "min"),
         last=("timestamp", "max"), source_files=("source_file", "nunique"))
    .reset_index()
)
display(coverage)
print(f"{len(combined_raw) - len(distinct):,} identical repeat rows collapsed")
print(f"{len(conflicts):,} records with conflicting timestamp keys set aside for review")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
combined.to_csv(OUTPUT_DIR / "soleco_load_profiles_combined.csv", index=False)
coverage.to_csv(OUTPUT_DIR / "soleco_series_coverage.csv", index=False)

conflicts.to_csv(OUTPUT_DIR / "soleco_conflicting_records.csv", index=False)
print(f"Saved outputs to {OUTPUT_DIR}")
# Compare each sheet before and after filtering overlapping measurements.
source_key = ["source_file", "source_sheet"]
parsed_kw = combined_raw.loc[combined_raw.KW_DEL.notna()]
retained_kw = combined.loc[combined.KW_DEL.notna()]
parsed_sheet = parsed_kw.groupby(source_key).timestamp.agg(
    parsed_kw="size", first_kw="min", last_kw="max",
)
retained_sheet = retained_kw.groupby(source_key).size().rename("retained_kw")
gap_stop = GAP_END + pd.Timedelta(days=1)
gap_parsed = parsed_kw.loc[parsed_kw.timestamp.between(GAP_START, gap_stop, inclusive="left")]
gap_retained = retained_kw.loc[retained_kw.timestamp.between(GAP_START, gap_stop, inclusive="left")]
gap_sheet = gap_parsed.groupby(source_key).size().rename("gap_parsed_kw")
gap_retained_sheet = gap_retained.groupby(source_key).size().rename("gap_retained_kw")
gap_audit = import_audit.join(parsed_sheet, on=source_key).join(
    retained_sheet, on=source_key,
).join(gap_sheet, on=source_key).join(gap_retained_sheet, on=source_key)
gap_years = "|".join(str(year) for year in range(GAP_START.year, GAP_END.year + 1))
dated_path = gap_audit.source_file.str.contains(rf"(?<!\d)(?:{gap_years})(?!\d)", regex=True)
gap_audit = gap_audit.loc[dated_path | gap_audit.gap_parsed_kw.notna()]
gap_audit.to_csv(OUTPUT_DIR / "soleco_gap_workbook_audit.csv", index=False)
gap_fields = [*source_key, "source_format", "status", "import_status", "invalid_timestamp_rows",
              "first_kw", "last_kw", "gap_parsed_kw", "gap_retained_kw"]
display(gap_audit[gap_fields])
gap_months = pd.period_range(GAP_START, GAP_END, freq="M")
gap_counts = pd.DataFrame(index=gap_months)
for stage, readings in [("Parsed readings", gap_parsed), ("Retained readings", gap_retained)]:
    counts = readings.groupby(readings.timestamp.dt.to_period("M")).size()
    gap_counts[stage] = counts.reindex(gap_months, fill_value=0)
display(gap_counts)


## 4. Daily delivered load

Plot every series with recorded `KW_DEL`. Use the total in its own panel if available. Hover reports the number of readings underlying each daily median. Generic wide-profile measurements remain in the export until their units are confirmed.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.colors import qualitative

load = combined.loc[combined.KW_DEL.notna(), ["timestamp", "series_id", "KW_DEL"]].copy()
if load.empty:
    raise ValueError("The archive has no numeric KW_DEL readings. Inspect coverage and wide-format reported_load units.")
station_order = sorted(load.series_id.unique(), key=lambda s: (s == "TOTAL", s))
station_colors = {s: qualitative.Dark24[i % 24] for i, s in enumerate(station_order)}
station_colors["TOTAL"] = "#303840"
daily_load = load.groupby(["series_id", pd.Grouper(key="timestamp", freq="D")]).KW_DEL.agg(["median", "count"])
dates = pd.date_range(load.timestamp.min().normalize(), load.timestamp.max().normalize(), freq="D")
daily_load = daily_load.reindex(pd.MultiIndex.from_product([station_order, dates], names=["series_id", "date"]))
daily_load["count"] = daily_load["count"].fillna(0).astype(int)

style = dict(template="plotly_white", paper_bgcolor="rgba(0,0,0,0)", font=dict(size=14),
             margin=dict(l=80, r=25, t=55, b=55),
             legend=dict(orientation="h", x=0, y=1.02, yanchor="bottom"))
separate_total = "TOTAL" in station_order and len(station_order) > 1
fig_daily = make_subplots(rows=2 if separate_total else 1, cols=1, shared_xaxes=True)
for station in station_order:
    part = daily_load.loc[station]
    row = 2 if separate_total and station == "TOTAL" else 1
    fig_daily.add_trace(go.Scatter(
        x=part.index, y=part["median"], name=station, mode="lines+markers",
        line=dict(color=station_colors[station], width=1.3), marker=dict(size=2),
        connectgaps=False, customdata=part["count"],
        hovertemplate="%{x|%d %b %Y}<br>%{y:,.1f} kW · %{customdata} readings<extra>%{fullData.name}</extra>",
    ), row=row, col=1)
fig_daily.update_layout(**style, height=560 if separate_total else 400)
fig_daily.update_yaxes(title_text="Median load (kW)", row=1, col=1)
if separate_total:
    fig_daily.update_yaxes(title_text="Total load (kW)", row=2, col=1)
fig_daily.update_xaxes(title_text="Date", row=2 if separate_total else 1, col=1)
fig_daily.show()

## 5. Monthly availability by meter

Each cell shows the percentage of calendar days with at least one delivered-load reading. Hover gives the day count and total valid readings. A partial day counts once. The denominator is the full calendar month, including the first and last months of the archive.

In [ ]:
day_counts = daily_load["count"].unstack("series_id")
days_present = day_counts.gt(0).resample("MS").sum()
monthly_readings = day_counts.resample("MS").sum()
pct = days_present.div(days_present.index.days_in_month, axis=0).mul(100)
fig_coverage = go.Figure(go.Heatmap(
    x=pct.index, y=pct.columns, z=pct.T.to_numpy(), zmin=0, zmax=100,
    colorscale=[[0, "#EBEEF2"], [0.5, "#91C9AD"], [1, "#146540"]],
    colorbar=dict(title="Days (%)"),
    customdata=np.stack([days_present.T, monthly_readings.T], axis=-1),
    hovertemplate="%{y} · %{x|%b %Y}<br>%{customdata[0]:.0f} days (%{z:.1f}%)<br>%{customdata[1]:,.0f} readings<extra></extra>",
))
fig_coverage.update_layout(**style, height=max(275, 32 * len(station_order) + 110))
fig_coverage.update_yaxes(autorange="reversed", type="category")
fig_coverage.update_xaxes(title_text="Month")
fig_coverage.show()

## 6. Inspect native readings

Choose a week to inspect the original delivered-load readings. The default opens the week containing the August 2022 worksheet example. Where that week is unavailable, the first observed week is used. Lines break across intervals longer than 15 minutes. Every recorded point remains visible.

In [ ]:
week_start = pd.Timestamp("2022-07-26")
if not load.timestamp.between(week_start, week_start + pd.Timedelta(days=7), inclusive="left").any():
    week_start = dates[0]
week_end = week_start + pd.Timedelta(days=7)
week = load.loc[load.timestamp.between(week_start, week_end, inclusive="left")]
fig_week = go.Figure()
for station, part in week.groupby("series_id"):
    part = part.sort_values("timestamp").set_index("timestamp")["KW_DEL"]
    # Insert one missing point inside each gap without generating a dense time grid.
    after_gap = part.index.to_series().diff().gt(pd.Timedelta(minutes=15))
    breaks = part.index[after_gap] - pd.Timedelta(seconds=1)
    part = part.reindex(part.index.union(breaks).sort_values())
    fig_week.add_trace(go.Scatter(
        x=part.index, y=part, name=station, mode="lines+markers", connectgaps=False,
        line=dict(color=station_colors[station], width=1.3), marker=dict(size=3),
        hovertemplate="%{x|%d %b %Y %H:%M}<br>%{y:,.3f} kW<extra>%{fullData.name}</extra>",
    ))
fig_week.update_layout(**style, height=420)
fig_week.update_xaxes(title_text="Date and time", range=[week_start, week_end], tickformat="%d %b")
fig_week.update_yaxes(title_text="Delivered load (kW)")
fig_week.show()